<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/ResNet-18/Prune_ResNet18_CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import copy
import torch
import torchvision
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Replace ResNet Classifier

In [19]:
def replace_resnet_classifier(model, num_classes):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

## ImageNet-style Transforms

In [20]:
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=16),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

## Load the CIFAR-100 DATASET

In [21]:
# Get transforms
train_transform, eval_transform = get_transforms(img_size=224)

# Load CIFAR-100 training set twice:
# once with train augmentation, once with eval transform
full_train_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

full_train_dataset_eval = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=False,
    transform=eval_transform
)

# Official test set
testset = torchvision.datasets.CIFAR100(
    root='./data',
    train=False,
    download=True,
    transform=eval_transform
)

# Split original training set into train and validation
total_size = len(full_train_dataset)
val_size = int(0.1 * total_size)   # 10% validation
train_size = total_size - val_size

generator = torch.Generator().manual_seed(42)

train_subset_tmp, val_subset_tmp = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)

train_indices = train_subset_tmp.indices
val_indices = val_subset_tmp.indices

trainset = Subset(full_train_dataset, train_indices)
valset = Subset(full_train_dataset_eval, val_indices)

# DataLoaders
trainloader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

valloader = DataLoader(
    valset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

print(f"Train samples: {len(trainset)}")
print(f"Validation samples: {len(valset)}")
print(f"Test samples: {len(testset)}")
print("CIFAR-100 dataset loaded successfully.")

Train samples: 45000
Validation samples: 5000
Test samples: 10000
CIFAR-100 dataset loaded successfully.


# Evaluation Functions

Defines reusable functions to evaluate the model on validation/test data.

In [22]:
def train_one_epoch(model, trainloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

# Collect Prunable Parameters

Finds all convolution and linear layers to prune.

In [23]:
def get_prunable_parameters(model):
    parameters_to_prune = []

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))

    return parameters_to_prune

# Apply Global Unstructured Pruning

Prunes a given fraction of weights across the whole model.

In [24]:
def apply_global_pruning(model, amount):
    parameters_to_prune = get_prunable_parameters(model)

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )

    return model

# Make Pruning Permanent

Removes pruning reparameterization so the pruned weights become permanent.

In [25]:
def remove_pruning_reparam(model):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, "weight")
            except ValueError:
                pass
    return model

# Measure Sparsity

Prints how many weights are zero after pruning.

In [26]:
def measure_sparsity(model):
    zero_params = 0
    total_params = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.data
            zero_params += torch.sum(weight == 0).item()
            total_params += weight.numel()

    sparsity = 100.0 * zero_params / total_params
    print(f"Global sparsity: {sparsity:.2f}%")
    return sparsity

# Fine-Tune Pruned Model


Fine-tunes the pruned model and saves the best recovered checkpoint based on validation accuracy.

In [27]:
def finetune_pruned_model(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    save_path="best_pruned_model_cifar100_resnet18.pth"
):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"New best pruned model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Optimizer for Pruned Model Recovery

Creates a conservative optimizer for recovery after pruning.

In [28]:
def get_pruned_model_optimizer(model):
    return torch.optim.SGD(
        model.parameters(),
        lr=1e-4,
        momentum=0.9,
        weight_decay=1e-4
    )

# Scheduler for Recovery

Reduces learning rate during recovery training.

In [29]:
def get_pruned_model_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Run One Pruning Experiment

Loads the clean model, prunes it, fine-tunes it, and reports final recovered performance.

In [30]:
# prune_amount = 0.20   # try 0.1, 0.2, 0.3, 0.5
prune_amount_list = [0.20, 0.40, 0.60, 0.80]
criterion = nn.CrossEntropyLoss()

for prune_amount in prune_amount_list:
    print(f"\nFine-tuning pruned model with prune_amount={prune_amount}...\n")
    model = models.resnet18(weights=None)
    model = replace_resnet_classifier(model, num_classes=100)
    model.load_state_dict(torch.load("drive/MyDrive/DLBA Project/Models/finetuned_resnet18_cifar100_base_model.pth", map_location=device))
    model = model.to(device)

    # evaluate clean checkpoint
    clean_val_loss, clean_val_acc = evaluate(model, valloader, criterion, device)
    print(f"Clean Val Acc before pruning: {clean_val_acc:.2f}%")

    # prune
    model = apply_global_pruning(model, amount=prune_amount)
    measure_sparsity(model)

    # make pruning permanent
    model = remove_pruning_reparam(model)

    # evaluate immediately after pruning
    pruned_val_loss, pruned_val_acc = evaluate(model, valloader, criterion, device)
    print(f"Val Acc right after pruning: {pruned_val_acc:.2f}%")

    # fine-tune pruned model
    optimizer = get_pruned_model_optimizer(model)
    scheduler = get_pruned_model_scheduler(optimizer)

    history, best_val_acc = finetune_pruned_model(
        model=model,
        trainloader=trainloader,
        valloader=valloader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=8,
        scheduler=scheduler,
        best_val_acc=0.0,
        save_path=f"drive/MyDrive/DLBA Project/Models/pruned_resnet18_cifar100_{int(prune_amount*100)}.pth"
    )

    print(f"Best recovered Val Acc: {best_val_acc:.2f}%")


Fine-tuning pruned model with prune_amount=0.2...

Clean Val Acc before pruning: 77.70%
Global sparsity: 20.00%
Val Acc right after pruning: 77.14%
New best pruned model saved with val acc: 77.50%
Epoch [1/8] | Train Loss: 0.5471 | Train Acc: 83.69% | Val Loss: 0.7255 | Val Acc: 77.50%
New best pruned model saved with val acc: 77.82%
Epoch [2/8] | Train Loss: 0.5296 | Train Acc: 84.02% | Val Loss: 0.7194 | Val Acc: 77.82%
Epoch [3/8] | Train Loss: 0.5163 | Train Acc: 84.43% | Val Loss: 0.7131 | Val Acc: 77.48%
New best pruned model saved with val acc: 77.98%
Epoch [4/8] | Train Loss: 0.4957 | Train Acc: 85.16% | Val Loss: 0.7095 | Val Acc: 77.98%
New best pruned model saved with val acc: 78.08%
Epoch [5/8] | Train Loss: 0.4918 | Train Acc: 85.39% | Val Loss: 0.7053 | Val Acc: 78.08%
New best pruned model saved with val acc: 78.16%
Epoch [6/8] | Train Loss: 0.4940 | Train Acc: 85.11% | Val Loss: 0.7022 | Val Acc: 78.16%
Epoch [7/8] | Train Loss: 0.4883 | Train Acc: 85.50% | Val Loss: 0

# Final Test Evaluation for Best Pruned Model

Loads the best recovered pruned model and evaluates it on the test set.

In [31]:
for prune_amount in prune_amount_list:
    print(f"\nEvaluating best pruned model with {int(prune_amount*100)}% pruning:")
    best_pruned_model = models.resnet18(weights=None)
    best_pruned_model = replace_resnet_classifier(best_pruned_model, num_classes=100)
    best_pruned_model.load_state_dict(
        torch.load(f"drive/MyDrive/DLBA Project/Models/pruned_resnet18_cifar100_{int(prune_amount*100)}.pth", map_location=device)
    )
    best_pruned_model = best_pruned_model.to(device)

    test_loss, test_acc = evaluate(best_pruned_model, testloader, criterion, device)
    print(f"Pruned Model Test Loss: {test_loss:.4f}")
    print(f"Pruned Model Test Acc: {test_acc:.2f}%")


Evaluating best pruned model with 20% pruning:
Pruned Model Test Loss: 0.7239
Pruned Model Test Acc: 78.05%

Evaluating best pruned model with 40% pruning:
Pruned Model Test Loss: 0.7294
Pruned Model Test Acc: 77.96%

Evaluating best pruned model with 60% pruning:
Pruned Model Test Loss: 0.7463
Pruned Model Test Acc: 77.32%

Evaluating best pruned model with 80% pruning:
Pruned Model Test Loss: 0.8249
Pruned Model Test Acc: 75.10%
